```markdown
# Engenharia de Atributos (Feature Engineering) com Dados de Crédito

Este notebook apresenta um guia prático e passo a passo sobre como preparar dados reais para algoritmos de Machine Learning. Você aprenderá a tratar valores ausentes, lidar com valores discrepantes (*outliers*), converter variáveis de texto em formatos numéricos e normalizar escalas.

---

## 🎯 Objetivos de Aprendizagem
Ao final deste estudo, você será capaz de:
1. **Identificar e tratar valores nulos** de forma apropriada usando métricas estatísticas robustas como a mediana.
2. **Detectar e tratar *outliers*** (valores discrepantes) aplicando o desvio padrão como limite de corte.
3. **Transformar variáveis categóricas** (texto) em representações numéricas usando as técnicas de *Label Encoding* e *One-Hot Encoding* (*get_dummies*).
4. **Padronizar features numéricas** usando o *StandardScaler* para garantir que todas as variáveis tenham o mesmo peso no modelo.
5. **Manipular dados temporais** para extrair novas informações valiosas (Ano, Mês e Dia da Semana).

---

## 📚 Pré-requisitos e Bibliotecas Utilizadas
Para executar este notebook, utilizamos as seguintes bibliotecas do ecossistema Python:
* **Pandas (`pd`)**: A biblioteca fundamental para manipulação e análise de dados. Usada para ler o arquivo CSV, tratar nulos, criar variáveis e limpar o dataset.
* **Scikit-Learn (Sklearn)**:
  * `LabelEncoder`: Usado para converter textos ordenados ou categorias simples em números inteiros (ex: Segunda-feira -> 0, Terça-feira -> 1).
  * `StandardScaler`: Usado para realizar a padronização dos dados numéricos (escala), deixando-os com média 0 e desvio padrão 1, evitando que variáveis com números muito grandes distorçam as previsões.
```

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

```markdown
### Passo 1: Importação de Bibliotecas
**O que este passo faz:** Importa as ferramentas essenciais que utilizaremos ao longo de todo o projeto.

**Por que é necessário:** Antes de construir qualquer pipeline, precisamos garantir que o Python tenha acesso aos pacotes de manipulação de dados (`pandas`) e de pré-processamento de dados do `scikit-learn`.
```

In [2]:
dataset = pd.read_csv('credit_simple.csv', sep=';')
dataset.shape

(1000, 8)

```markdown
### Passo 2: Carregamento do Dataset e Verificação de Dimensões
**O que este passo faz:** Carrega a base de dados `credit_simple.csv` (usando o ponto e vírgula como separador) e exibe o formato da tabela (`shape`).

**Por que é necessário:** Precisamos saber o tamanho do nosso problema. O resultado nos mostra quantas linhas (registros) e colunas (atributos) temos para trabalhar.

**Como interpretar a saída:** A saída `(1000, 8)` indica que temos exatamente 1000 clientes cadastrados e 8 colunas de informações para cada um deles.
```

In [3]:
dataset.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,CLASSE
0,1169.0,4,67,nenhum,01/01/2019,masculino solteiro,radio/tv,bom
1,5951.0,2,22,nenhum,01/01/2020,fem div/cas,radio/tv,ruim
2,2096.0,3,49,nenhum,02/01/2020,masculino solteiro,educação,bom
3,7882.0,4,45,nenhum,02/01/2019,masculino solteiro,mobilia/equipamento,bom
4,4870.0,4,53,nenhum,03/01/2018,masculino solteiro,carro novo,ruim


```markdown
### Passo 3: Visualização das Primeiras Linhas
**O que este passo faz:** Exibe as 5 primeiras linhas da tabela.

**Por que é necessário:** Permite um primeiro contato visual com os dados. Conseguimos ver os nomes das colunas e os tipos de dados que cada uma carrega (números, datas, textos).

**Como interpretar a saída:** Observamos variáveis como `SALDO_ATUAL` (numérico), `DATA` (data em formato texto), `ESTADOCIVIL` (categoria em texto) e `CLASSE` (nossa variável alvo, que indica se o cliente é 'bom' ou 'ruim' pagador).
```

In [4]:
y = dataset['CLASSE']
X = dataset.iloc[:,:-1]

```markdown
### Passo 4: Divisão de Features (X) e Target (y)
**O que este passo faz:** Separa a nossa variável que queremos prever (`CLASSE`, guardada em `y`) de todas as outras colunas que usaremos para aprender (`X`).

**Por que é necessário:** Modelos de Machine Learning supervisionados exigem essa separação clara: $X$ representa as perguntas (características do cliente) e $y$ representa a resposta correta (se ele pagou ou não).
```

In [5]:
X.isnull().sum()

,0
SALDO_ATUAL,7
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,8
PROPOSITO,0


```markdown
### Passo 5: Verificação de Valores Nulos (Missing Values)
**O que este passo faz:** Contabiliza quantos valores em branco (ausentes) existem em cada coluna.

**Por que é necessário:** A maioria dos algoritmos de inteligência artificial não aceita valores nulos. Se tentarmos treinar um modelo com dados ausentes, ocorrerá um erro catastrófico.

**Como interpretar a saída:** Descobrimos que `SALDO_ATUAL` possui 7 valores nulos e `ESTADOCIVIL` possui 8 valores nulos. As demais colunas estão completas.
```

In [6]:
mediana = X['SALDO_ATUAL'].median()
mediana

2323.0

```markdown
### Passo 6: Cálculo da Mediana para Preenchimento
**O que este passo faz:** Calcula a mediana da coluna `SALDO_ATUAL`.

**Por que é necessário:** A **mediana** é o valor que fica exatamente no centro dos dados ordenados. Ela é preferível à média em distribuições com valores muito extremos, pois não é distorcida por eles.

**Como interpretar a saída:** O valor `2323.0` será usado para substituir as lacunas vazias dessa coluna.
```

In [7]:
X['SALDO_ATUAL'].fillna(mediana,inplace=True)
X.isnull().sum()

/tmp/ipykernel_5563/2913618785.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['SALDO_ATUAL'].fillna(mediana,inplace=True)


,0
SALDO_ATUAL,0
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,8
PROPOSITO,0


```markdown
### Passo 7: Tratamento de Nulos na Variável Numérica
**O que este passo faz:** Substitui os valores nulos de `SALDO_ATUAL` pela mediana calculada e valida se os nulos foram zerados.

**Por que é necessário:** Garante que a coluna numérica não tenha mais nenhuma ausência de dados.

**💡 Para pensar:** O uso de `inplace=True` em atribuições encadeadas ou diretamente em colunas está depreciado nas versões recentes do Pandas e gerará avisos de *FutureWarning*. O ideal no futuro é reatribuir diretamente: `X['SALDO_ATUAL'] = X['SALDO_ATUAL'].fillna(mediana)`.
```

In [8]:
agrupado = X.groupby(['ESTADOCIVIL']).size()
agrupado

,0
ESTADOCIVIL,
fem div/cas,308
masculino casado/viuvo,92
masculino div/sep,50
masculino solteiro,542


```markdown
### Passo 8: Análise de Frequência da Variável Categórica
**O que este passo faz:** Agrupa a coluna `ESTADOCIVIL` para ver quais categorias aparecem mais vezes e qual a contagem de cada uma.

**Por que é necessário:** Para tratar os nulos de uma coluna de texto (categórica), uma estratégia comum é utilizar a **moda** (a categoria que mais se repete).

**Como interpretar a saída:** A categoria `masculino solteiro` é a mais comum, aparecendo 542 vezes.
```

In [9]:
X['ESTADOCIVIL'].fillna('masculino solteiro', inplace=True)
X.isnull().sum()

/tmp/ipykernel_5563/1274388989.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X['ESTADOCIVIL'].fillna('masculino solteiro', inplace=True)


,0
SALDO_ATUAL,0
RESIDENCIADESDE,0
IDADE,0
OUTROSPLANOSPGTO,0
DATA,0
ESTADOCIVIL,0
PROPOSITO,0


```markdown
### Passo 9: Imputação pela Moda
**O que este passo faz:** Substitui as lacunas vazias de `ESTADOCIVIL` pela moda calculada ('masculino solteiro') e valida que não restam mais valores nulos no dataset.
```

In [10]:
desv = X['SALDO_ATUAL'].std()
desv

685936688.9820064

```markdown
### Passo 10: Investigação de Outliers (Anomalias)
**O que este passo faz:** Calcula o desvio padrão (`std`) da coluna `SALDO_ATUAL`.

**Por que é necessário:** O desvio padrão mede a dispersão dos dados. Valores excessivamente altos fora de uma faixa esperada (como várias vezes o desvio padrão) podem ser erros de digitação ou casos extremamente atípicos que atrapalham o modelo.

**Como interpretar a saída:** O desvio padrão é um número gigante (`685936688.98`), o que acende um alerta vermelho sobre a existência de valores absurdos de saldo na nossa base.
```

In [11]:
X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL']

,SALDO_ATUAL
127,2.541111e+09
160,2.154441e+10


```markdown
### Passo 11: Localização dos Outliers
**O que este passo faz:** Filtra e exibe os registros onde o `SALDO_ATUAL` é maior ou igual a duas vezes o desvio padrão.

**Como interpretar a saída:** Encontramos dois registros (índices 127 e 160) com saldos na casa dos bilhões! Claramente são erros de entrada ou ruídos graves no banco de dados.
```

In [12]:
mediana = X['SALDO_ATUAL'].median()
mediana

2323.0

```markdown
### Passo 12: Recálculo da Mediana
**O que este passo faz:** Obtém novamente o valor mediano da coluna para servir de substituto seguro para estes erros.
```

In [13]:
X.loc[X['SALDO_ATUAL']>= 2 * desv, 'SALDO_ATUAL'] = mediana
X.loc[X['SALDO_ATUAL']>= 2 * desv]

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO


```markdown
### Passo 13: Substituição de Outliers
**O que este passo faz:** Substitui os valores absurdos encontrados pela mediana estável de `2323.0`.

**Por que é necessário:** Ao remover esses valores extremos bizarros, evitamos que o algoritmo de aprendizado de máquina ajuste suas regras baseado em erros grosseiros de digitação.
```

In [14]:
agrupado = X.groupby(['PROPOSITO']).size()
agrupado

,0
PROPOSITO,
Eletrodomésticos,12
carro novo,234
carro usado,103
educação,50
mobilia/equipamento,181
negócios,97
obras,22
outros,12
qualificação,9


```markdown
### Passo 14: Análise da Variável 'PROPOSITO'
**O que este passo faz:** Conta a frequência das categorias na coluna `PROPOSITO`.

**Por que é necessário:** Categorias com pouquíssimas amostras podem causar *overfitting* ou dificultar o aprendizado do algoritmo. Agrupá-las sob uma categoria geral (como 'outros') ajuda a simplificar o modelo.
```

In [15]:
X.loc[X['PROPOSITO']=='Eletrodomésticos','PROPOSITO'] = 'outros'
X.loc[X['PROPOSITO']=='qualificação','PROPOSITO'] = 'outros'
agrupado = X.groupby(['PROPOSITO']).size()
agrupado

,0
PROPOSITO,
carro novo,234
carro usado,103
educação,50
mobilia/equipamento,181
negócios,97
obras,22
outros,33
radio/tv,280


```markdown
### Passo 15: Agrupamento de Categorias Raras
**O que este passo faz:** Junta as categorias muito raras ('Eletrodomésticos' e 'qualificação') sob a categoria 'outros'.

**Como interpretar a saída:** Agora a categoria 'outros' passou a conter 33 ocorrências, simplificando nossa coluna de propósitos de crédito.
```

In [16]:
X['DATA']

,DATA
0,01/01/2019
1,01/01/2020
2,02/01/2020
3,02/01/2019
4,03/01/2018
...,...
995,29/06/2018
996,30/06/2018
997,03/07/2018
998,04/07/2019


```markdown
### Passo 16: Análise de Dados de Data
**O que este passo faz:** Exibe as datas da coluna `DATA` para analisar o formato original.

**Como interpretar a saída:** As datas estão no tipo `object` (texto) no formato brasileiro `dd/mm/aaaa`.
```

In [17]:
X['DATA'] = pd.to_datetime(X['DATA'], format='%d/%m/%Y')

```markdown
### Passo 17: Conversão para Datetime
**O que este passo faz:** Converte o tipo de texto para o tipo de dado de data oficial do Pandas (`datetime64`).

**Por que é necessário:** Textos comuns não possuem inteligência temporal. Convertendo para datetime, habilitamos o Pandas a extrair componentes cronológicos de forma automática.
```

In [18]:
X['DATA']

,DATA
0,2019-01-01
1,2020-01-01
2,2020-01-02
3,2019-01-02
4,2018-01-03
...,...
995,2018-06-29
996,2018-06-30
997,2018-07-03
998,2019-07-04


In [19]:
X['ANO'] = X['DATA'].dt.year
X['MES'] = X['DATA'].dt.month
X['DIASEMANA'] = X['DATA'].dt.day_name()

```markdown
### Passo 18: Engenharia de Recursos Temporais
**O que este passo faz:** Cria três novas colunas a partir da data: o Ano, o Mês e o Dia da Semana.

**Por que é necessário:** Algoritmos não conseguem aprender diretamente com uma data completa individual. Extraindo padrões como 'Dia da Semana', o algoritmo consegue entender se há mais inadimplência nas segundas ou sextas-feiras, por exemplo.
```

In [20]:
X['DIASEMANA']

,DIASEMANA
0,Tuesday
1,Wednesday
2,Thursday
3,Wednesday
4,Wednesday
...,...
995,Friday
996,Saturday
997,Tuesday
998,Thursday


In [21]:
X['ESTADOCIVIL'].unique()

array(['masculino solteiro', 'fem div/cas', 'masculino div/sep',
       'masculino casado/viuvo'], dtype=object)

In [22]:
X['PROPOSITO'].unique()

array(['radio/tv', 'educação', 'mobilia/equipamento', 'carro novo',
       'carro usado', 'negócios', 'outros', 'obras'], dtype=object)

In [23]:
X['DIASEMANA'].unique()

array(['Tuesday', 'Wednesday', 'Thursday', 'Saturday', 'Sunday', 'Monday',
       'Friday'], dtype=object)

In [24]:
labelencoder1 = LabelEncoder()
X['ESTADOCIVIL'] = labelencoder1.fit_transform(X['ESTADOCIVIL'])
X['PROPOSITO'] = labelencoder1.fit_transform(X['PROPOSITO'])
X['DIASEMANA'] = labelencoder1.fit_transform(X['DIASEMANA'])

```markdown
### Passo 19: Codificação Ordinal (Label Encoding)
**O que este passo faz:** Transforma as colunas de texto categórico (`ESTADOCIVIL`, `PROPOSITO`, `DIASEMANA`) em números de zero a N-1 usando o `LabelEncoder`.

**Por que é necessário:** Modelos matemáticos só processam equações numéricas. Esta técnica é ótima para colunas que possuem uma ordem intrínseca ou quando queremos economizar espaço de colunas.
```

In [25]:
X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5
1,5951.0,2,22,nenhum,2020-01-01,0,7,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4
3,7882.0,4,45,nenhum,2019-01-02,3,3,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6


In [26]:
outros = X['OUTROSPLANOSPGTO'].unique()
outros

array(['nenhum', 'banco', 'stores'], dtype=object)

In [27]:
z = pd.get_dummies(X['OUTROSPLANOSPGTO'], prefix = 'OUTROS')

```markdown
### Passo 20: Codificação One-Hot (Dummies)
**O que este passo faz:** Cria colunas binárias (`0` ou `1`) para cada categoria da coluna `OUTROSPLANOSPGTO`.

**Por que é necessário:** Quando as opções não possuem nenhuma ordem lógica entre si (como 'banco', 'stores', 'nenhum'), o *Label Encoding* pode confundir o algoritmo fazendo-o pensar que 'stores' (valor 2) é maior ou melhor que 'banco' (valor 1). O *One-Hot Encoding* resolve isso criando uma coluna de sim/não para cada opção.
```

In [28]:
z

,OUTROS_banco,OUTROS_nenhum,OUTROS_stores
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False
...,...,...,...
995,False,True,False
996,False,True,False
997,False,True,False
998,False,True,False


In [29]:
X

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5
1,5951.0,2,22,nenhum,2020-01-01,0,7,2020,1,6
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4
3,7882.0,4,45,nenhum,2019-01-02,3,3,2019,1,6
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6
...,...,...,...,...,...,...,...,...,...,...
995,1736.0,4,31,nenhum,2018-06-29,0,3,2018,6,0
996,3857.0,4,40,nenhum,2018-06-30,2,1,2018,6,2
997,804.0,4,38,nenhum,2018-07-03,3,7,2018,7,5
998,1845.0,4,23,nenhum,2019-07-04,3,7,2019,7,4


In [30]:
sc = StandardScaler()
m = sc.fit_transform(X.iloc[:,0:3])
m

array([[-0.74551643,  1.04698668,  1.6392759 ],
       [ 0.95774038, -0.76597727, -0.74024139],
       [-0.41533679,  0.14050471,  0.68746898],
       ...,
       [-0.87552244,  1.04698668,  0.1058092 ],
       [-0.50473818,  1.04698668, -0.68736323],
       [ 0.46799171,  1.04698668, -0.47585058]], shape=(1000, 3))

```markdown
### Passo 21: Padronização de Escala (Standardization)
**O que este passo faz:** Ajusta as três primeiras colunas de `X` (`SALDO_ATUAL`, `RESIDENCIADESDE`, `IDADE`) para que possuam média perto de 0 e desvio padrão igual a 1.

**Por que é necessário:** Repare que a idade varia de 18 a 80, enquanto o saldo varia de 100 a 10.000. Algoritmos baseados em distância (como KNN) achariam que o saldo é infinitamente mais importante do que a idade apenas porque os números são maiores. A padronização põe todos no mesmo patamar geométrico.
```

In [31]:
X = pd.concat([X,z,pd.DataFrame(m,columns=['SALDO_ATUAL_N','RESIDENCEADESDE_N','IDADE_N'])],axis=1)
X.head()

,SALDO_ATUAL,RESIDENCIADESDE,IDADE,OUTROSPLANOSPGTO,DATA,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA,OUTROS_banco,OUTROS_nenhum,OUTROS_stores,SALDO_ATUAL_N,RESIDENCEADESDE_N,IDADE_N
0,1169.0,4,67,nenhum,2019-01-01,3,7,2019,1,5,False,True,False,-0.745516,1.046987,1.639276
1,5951.0,2,22,nenhum,2020-01-01,0,7,2020,1,6,False,True,False,0.957740,-0.765977,-0.740241
2,2096.0,3,49,nenhum,2020-01-02,3,2,2020,1,4,False,True,False,-0.415337,0.140505,0.687469
3,7882.0,4,45,nenhum,2019-01-02,3,3,2019,1,6,False,True,False,1.645526,1.046987,0.475956
4,4870.0,4,53,nenhum,2018-01-03,3,0,2018,1,6,False,True,False,0.572709,1.046987,0.898982


```markdown
### Passo 22: Concatenação dos Dados Transformados
**O que este passo faz:** Junta as variáveis transformadas e normalizadas de volta ao nosso conjunto de dados original `X`.
```

In [32]:
X.drop(columns=['SALDO_ATUAL','RESIDENCIADESDE','IDADE','OUTROSPLANOSPGTO','DATA','OUTROS_banco'], inplace=True)
X

,ESTADOCIVIL,PROPOSITO,ANO,MES,DIASEMANA,OUTROS_nenhum,OUTROS_stores,SALDO_ATUAL_N,RESIDENCEADESDE_N,IDADE_N
0,3,7,2019,1,5,True,False,-0.745516,1.046987,1.639276
1,0,7,2020,1,6,True,False,0.957740,-0.765977,-0.740241
2,3,2,2020,1,4,True,False,-0.415337,0.140505,0.687469
3,3,3,2019,1,6,True,False,1.645526,1.046987,0.475956
4,3,0,2018,1,6,True,False,0.572709,1.046987,0.898982
...,...,...,...,...,...,...,...,...,...,...
995,0,3,2018,6,0,True,False,-0.543562,1.046987,-0.264338
996,2,1,2018,6,2,True,False,0.211898,1.046987,0.211566
997,3,7,2018,7,5,True,False,-0.875522,1.046987,0.105809
998,3,7,2019,7,4,True,False,-0.504738,1.046987,-0.687363


```markdown
### Passo 23: Descarte de Variáveis Antigas
**O que este passo faz:** Remove as colunas originais que já foram transformadas ou normalizadas para não duplicar informações.

**Por que é necessário:** Manter as colunas antigas e as novas juntas causaria redundância de dados (multicolinearidade), o que prejudica a performance e a interpretação de muitos algoritmos.

**Como interpretar a saída:** Agora temos um dataset `X` limpo, puramente numérico e perfeitamente escalado, pronto para ser enviado a qualquer algoritmo de Machine Learning de ponta!

---

## ⚠️ Erros Comuns e Dicas de Execução
1. **Ordem de Execução:** Se você tentar rodar o Passo 23 (`X.drop(...)`) duas vezes seguidas, receberá um erro de `KeyError`, pois as colunas já foram apagadas na primeira execução. Se der erro, reinicie o Kernel e execute tudo de cima para baixo.
2. **Inplace Warning:** O aviso rosa de *FutureWarning* no tratamento de nulos não impede a execução do código atual, mas é um alerta importante para atualizar o código no futuro.
3. **Cuidado com o vazamento de dados (*Data Leakage*):** Ao usar o `StandardScaler`, no mundo real você deve aplicar o `fit` apenas na base de treino e o `transform` na base de teste para evitar que dados futuros influenciem seu processamento.

---

## 📝 Resumo do Aprendizado
* Tratamos dados faltantes de forma estatística segura (mediana para numéricos, moda para categóricos).
* Limpamos *outliers* gritantes gerados por ruídos de escala.
* Agrupamos categorias raras para evitar o superajuste.
* Realizamos engenharia de recursos temporais a partir de datas brutas.
* Transformamos categorias de texto em números com *Label Encoding* e *One-Hot Encoding*.
* Colocamos todas as variáveis numéricas na mesma escala através da Padronização.

---

## 🏋️ Exercícios Propostos
1. **(Fácil)** Crie uma nova coluna categórica no dataset original chamada `FAIXA_ETARIA` baseando-se na coluna `IDADE` (ex: Menor de 30 anos, Entre 30 e 50 anos, Maior de 50 anos) e aplique o *One-Hot Encoding* nela.
2. **(Médio)** Em vez de usar `StandardScaler` (que usa média e desvio padrão), aplique o redimensionador `MinMaxScaler` do Sklearn nas colunas numéricas de modo que todos os valores fiquem estritamente entre 0 e 1. Compare visualmente as duas saídas.
3. **(Desafiador)** Simule um pipeline completo: Crie uma função chamada `preprocess_pipeline(raw_df)` que receba o DataFrame original bruto carregado do CSV e devolva a matriz $X$ perfeitamente limpa e tratada de forma automatizada.

---

## 🚀 Para ir Além
* Pesquise sobre **Imputação Robusta por KNN (KNNImputer)** para preencher nulos usando similaridade de vizinhos em vez de apenas a mediana geral.
* Pesquise sobre a diferença matemática profunda entre **Normalização (MinMax)** e **Padronização (Z-Score/StandardScaler)** e quando escolher cada uma.
* Estude sobre o conceito de **Multicolinearidade** e por que remover uma das colunas após o *One-Hot Encoding* (o argumento `drop_first=True` do Pandas) ajuda a evitá-la.
```